In [1]:
pip install speechbrain transformers huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.7/912.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install TTS librosa soundfile scipy torchaudio

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

In [2]:
%pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.0.0
    Uninstalling setuptools-84.0.0:
      Successfully uninstalled setuptools-84.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install torch torchaudio

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install coqui-tts speechbrain librosa soundfile scipy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import speechbrain
import librosa
import soundfile as sf
from TTS.api import TTS

print("PyTorch:", torch.__version__)
print("SpeechBrain:", speechbrain.__version__)
print("CUDA available:", torch.cuda.is_available())
print("All imports successful!")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0
SpeechBrain: 1.1.2
CUDA available: False
All imports successful!


In [5]:

import json
import re
import shutil
import tempfile
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch

from speechbrain.inference.classifiers import EncoderClassifier


# =========================================================
# VOCULT CONFIGURATION
# =========================================================

PROJECT_DIR = Path.home() / "vocult"

# Original audio extracted before WhisperX transcription.
SOURCE_AUDIO = PROJECT_DIR / "files" / "extracted_audio.wav"

# Manifest created by xtts_worker.py.
INPUT_MANIFEST = (
    PROJECT_DIR / "files" / "segments" / "segments_manifest.json"
)

# Keep original XTTS files untouched.
OUTPUT_DIR = PROJECT_DIR / "files" / "segments_emotion"
OUTPUT_MANIFEST = OUTPUT_DIR / "segments_manifest_emotion.json"

MODEL_SOURCE = "speechbrain/emotion-recognition-wav2vec2-IEMOCAP"
MODEL_DIR = PROJECT_DIR / "pretrained_models" / "speechbrain_emotion"

EMOTION_SR = 16000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Subtle post-processing presets.
# These are acoustic effects, not true emotional voice synthesis.
PRESETS = {
    "neutral": {"pitch": 0.0, "gain_db": 0.0},
    "happy":   {"pitch": 0.35, "gain_db": 0.5},
    "sad":     {"pitch": -0.35, "gain_db": -0.5},
    "angry":   {"pitch": 0.15, "gain_db": 0.5},
}


# =========================================================
# INITIALIZATION
# =========================================================

def load_classifier():
    print("Loading SpeechBrain emotion model on", DEVICE)

    return EncoderClassifier.from_hparams(
        source=MODEL_SOURCE,
        savedir=str(MODEL_DIR),
        run_opts={"device": DEVICE},
    )


def load_original_audio(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Original audio not found: {path}\n"
            "Set SOURCE_AUDIO to your actual extracted audio file."
        )

    audio, _ = librosa.load(
        str(path),
        sr=EMOTION_SR,
        mono=True,
    )

    if audio.size == 0:
        raise ValueError("Original audio is empty.")

    return audio.astype(np.float32)


def resolve_audio_path(value):
    path = Path(value).expanduser()

    if path.is_file():
        return path

    # Also support paths relative to the project directory.
    candidate = PROJECT_DIR / path

    if candidate.is_file():
        return candidate

    raise FileNotFoundError(f"Audio file not found: {value}")


# =========================================================
# EMOTION DETECTION
# =========================================================

def normalize_emotion(label):
    label = str(label).lower().strip()

    if "ang" in label:
        return "angry"
    if "hap" in label:
        return "happy"
    if "sad" in label:
        return "sad"
    if "neu" in label:
        return "neutral"

    return "neutral"


def detect_emotion(classifier, audio_segment):
    """
    Classify an original speech segment with SpeechBrain.

    The pretrained IEMOCAP model recognizes broad categories:
    neutral, happy, sad and angry.
    """

    if len(audio_segment) < int(0.4 * EMOTION_SR):
        return {
            "emotion": "neutral",
            "model_label": "too_short",
            "score": None,
        }

    audio_segment = np.asarray(
        audio_segment,
        dtype=np.float32,
    )

    with tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False,
    ) as temp:
        temp_path = Path(temp.name)

    try:
        sf.write(
            str(temp_path),
            audio_segment,
            EMOTION_SR,
        )

        with torch.no_grad():
            _, score, _, predicted_label = (
                classifier.classify_file(str(temp_path))
            )

        # SpeechBrain usually returns the predicted label
        # as a list containing a string.
        if isinstance(predicted_label, (list, tuple)):
            label = predicted_label[0]
        elif hasattr(predicted_label, "tolist"):
            label = predicted_label.tolist()
            if isinstance(label, list):
                label = label[0]
        else:
            label = predicted_label

        if hasattr(score, "detach"):
            score = score.detach().cpu().numpy()

        score_array = np.asarray(score).reshape(-1)
        raw_score = (
            float(score_array[0])
            if score_array.size
            else None
        )

        return {
            "emotion": normalize_emotion(label),
            "model_label": str(label),
            "score": raw_score,
        }

    finally:
        temp_path.unlink(missing_ok=True)


def get_source_segment(audio, start, end):
    start_sample = max(0, int(float(start) * EMOTION_SR))
    end_sample = min(len(audio), int(float(end) * EMOTION_SR))

    if end_sample <= start_sample:
        return np.array([], dtype=np.float32)

    return audio[start_sample:end_sample]


# =========================================================
# PROCESS XTTS WAV FILES
# =========================================================

def apply_emotion_effect(audio, sample_rate, emotion):
    """
    Add conservative acoustic adjustments to generated audio.

    For genuine emotional prosody, use an emotion-conditioned
    TTS model or a model trained for expressive speech.
    """

    audio = np.asarray(audio, dtype=np.float32).copy()

    settings = PRESETS.get(emotion, PRESETS["neutral"])

    pitch = settings["pitch"]

    if pitch:
        audio = librosa.effects.pitch_shift(
            y=audio,
            sr=sample_rate,
            n_steps=pitch,
        )

    gain = 10 ** (settings["gain_db"] / 20.0)
    audio *= gain

    # Avoid clipping.
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0

    if peak > 0.98:
        audio *= 0.98 / peak

    return audio.astype(np.float32)


def get_source_times(item, index):
    """
    Support the manifest format generated by your XTTS notebook.
    """
    start = item.get("start")
    end = item.get("end")

    if start is None or end is None:
        raise ValueError(
            f"Manifest item {index} is missing start/end timestamps."
        )

    return float(start), float(end)


def process_manifest():
    if not INPUT_MANIFEST.is_file():
        raise FileNotFoundError(
            f"XTTS manifest not found: {INPUT_MANIFEST}\n"
            "Run xtts_worker.py first."
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    with INPUT_MANIFEST.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    # Accept either a list or {"segments": [...]}.
    if isinstance(manifest, dict):
        items = manifest.get("segments", manifest.get("items", []))
    elif isinstance(manifest, list):
        items = manifest
    else:
        raise ValueError("Unsupported manifest JSON structure.")

    if not items:
        raise ValueError("The XTTS manifest contains no segments.")

    original_audio = load_original_audio(SOURCE_AUDIO)
    classifier = load_classifier()

    updated_items = []
    errors = []

    for index, item in enumerate(items, start=1):
        print(f"\nProcessing segment {index}/{len(items)}")

        try:
            start, end = get_source_times(item, index)

            # Your existing XTTS worker uses audio_path.
            # Support common alternative manifest keys too.
            audio_value = (
                item.get("audio_path")
                or item.get("generated_audio")
                or item.get("path")
            )

            if not audio_value:
                raise ValueError("No audio_path in manifest item.")

            xtts_path = resolve_audio_path(audio_value)

            source_segment = get_source_segment(
                original_audio,
                start,
                end,
            )

            emotion_info = detect_emotion(
                classifier,
                source_segment,
            )

            emotion = emotion_info["emotion"]

            # Preserve the generated XTTS clip's original sample rate.
            generated, sample_rate = sf.read(
                str(xtts_path),
                dtype="float32",
                always_2d=False,
            )

            if generated.ndim == 2:
                # Preserve stereo if XTTS generated stereo.
                channels_first = False
                mono = generated.mean(axis=1)
            else:
                channels_first = False
                mono = generated

            # Apply effects to mono audio.
            processed = apply_emotion_effect(
                mono,
                sample_rate,
                emotion,
            )

            # Use a separate output directory so original clips
            # can always be recovered.
            output_name = (
                f"segment_{index:03d}_{emotion}.wav"
            )
            output_path = OUTPUT_DIR / output_name

            sf.write(
                str(output_path),
                processed,
                sample_rate,
            )

            result = dict(item)
            result.update({
                "detected_emotion": emotion,
                "emotion_model_label": emotion_info["model_label"],
                "emotion_model_score": emotion_info["score"],
                "original_xtts_audio_path": str(xtts_path),
                "audio_path": str(output_path),
            })

            updated_items.append(result)

            print("Emotion:", emotion)
            print("Saved:", output_path)

        except Exception as exc:
            errors.append({
                "segment": index,
                "error": repr(exc),
            })
            print("FAILED:", exc)

            # Preserve the manifest record for debugging.
            failed_item = dict(item)
            failed_item["emotion_processing_error"] = repr(exc)
            updated_items.append(failed_item)

    # Keep the original top-level JSON metadata when possible.
    if isinstance(manifest, dict):
        output_data = dict(manifest)
        output_data["segments"] = updated_items
    else:
        output_data = updated_items

    with OUTPUT_MANIFEST.open("w", encoding="utf-8") as f:
        json.dump(
            output_data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("\n========== VOCULT EMOTION REPORT ==========")
    print("Total segments:", len(items))
    print("Successful:", len(items) - len(errors))
    print("Failed:", len(errors))
    print("Output folder:", OUTPUT_DIR)
    print("Updated manifest:", OUTPUT_MANIFEST)

    if errors:
        print("\nErrors:")
        print(json.dumps(errors, indent=2, ensure_ascii=False))

    return output_data


if __name__ == "__main__":
    process_manifest()


FileNotFoundError: XTTS manifest not found: /Users/chaitanyabhardwaj/vocult/files/segments/segments_manifest.json
Run xtts_worker.py first.

In [6]:
from pathlib import Path

PROJECT_DIR = Path.home() / "vocult"

print("Project directory:", PROJECT_DIR)
print("Project exists:", PROJECT_DIR.exists())

for pattern in [
    "*manifest*.json",
    "*.wav",
    "*.json",
]:
    print(f"\n--- {pattern} ---")
    for path in PROJECT_DIR.rglob(pattern):
        if any(
            part.startswith(".") or part == "__pycache__"
            for part in path.parts
        ):
            continue
        print(path)

Project directory: /Users/chaitanyabhardwaj/vocult
Project exists: True

--- *manifest*.json ---

--- *.wav ---
/Users/chaitanyabhardwaj/vocult/files/xtts_test.wav
/Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_018.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_030.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_024.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_025.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_031.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_019.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_027.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_033.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_032.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_026.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_022.wav
/Users/chaitanyabhardwaj/vocult/files/segments/segment_036.wav
/Users/chaitanyabhar

In [7]:
from pathlib import Path

for folder in [
    Path.home() / "vocult" / "files" / "segments",
    Path.home() / "vocult" / "files",
    Path.home() / "vocult" / "segments",
]:
    print(f"\n{folder}")
    if folder.exists():
        for item in list(folder.iterdir())[:30]:
            print(" ", item.name)
    else:
        print("  Folder does not exist")


/Users/chaitanyabhardwaj/vocult/files/segments
  segment_018.wav
  segment_030.wav
  segment_024.wav
  segment_025.wav
  segment_031.wav
  segment_019.wav
  segment_027.wav
  segment_033.wav
  segment_032.wav
  segment_026.wav
  segment_022.wav
  segment_036.wav
  segment_037.wav
  segment_023.wav
  segment_035.wav
  segment_021.wav
  segment_009.wav
  segment_008.wav
  segment_020.wav
  segment_034.wav
  .gitkeep
  segment_041.wav
  segment_040.wav
  segment_042.wav
  segment_039.wav
  .ipynb_checkpoints
  segment_011.wav
  segment_005.wav
  segment_004.wav
  segment_010.wav

/Users/chaitanyabhardwaj/vocult/files
  tts_segments
  test_audio_text.json
  xtts_test.wav
  test.mp4
  output
  speaker_reference.wav
  test_audio_text_translated.json
  .ipynb_checkpoints
  test_audio.mp3
  segments
  test_video.mp4

/Users/chaitanyabhardwaj/vocult/segments
  Folder does not exist


In [9]:
from pathlib import Path
import subprocess

PROJECT = Path.home() / "vocult"

candidates = [
    PROJECT / "files" / "test_video.mp4",
    PROJECT / "files" / "test.mp4",
    PROJECT / "files" / "test_audio.mp3",
    PROJECT / "files" / "extracted_audio.wav",
]

for path in candidates:
    if not path.exists():
        print(f"MISSING: {path}")
        continue

    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "stream=codec_type",
            "-of", "csv=p=0",
            str(path),
        ],
        capture_output=True,
        text=True,
    )

    streams = result.stdout.strip().splitlines()
    print(f"{path.name}: {streams or 'No audio/video streams detected'}")

test_video.mp4: ['video']
test.mp4: ['video', 'audio']
test_audio.mp3: ['audio']
MISSING: /Users/chaitanyabhardwaj/vocult/files/extracted_audio.wav


In [10]:
from pathlib import Path
import subprocess

PROJECT = Path.home() / "vocult"
FILES = PROJECT / "files"

# This MP4 contains both video and audio.
INPUT_VIDEO = FILES / "test.mp4"

# Original audio used for speech/emotion analysis.
SOURCE_AUDIO = FILES / "test_audio.mp3"

OUTPUT_DIR = FILES / "emotion_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def has_audio(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=codec_type",
            "-of", "csv=p=0",
            str(path)
        ],
        capture_output=True,
        text=True
    )
    return result.returncode == 0 and bool(result.stdout.strip())

print("Input video:", INPUT_VIDEO, INPUT_VIDEO.exists())
print("Video contains audio:", has_audio(INPUT_VIDEO))
print("Source audio:", SOURCE_AUDIO, SOURCE_AUDIO.exists())
print("Audio file is valid:", has_audio(SOURCE_AUDIO))
print("Output directory:", OUTPUT_DIR)

Input video: /Users/chaitanyabhardwaj/vocult/files/test.mp4 True
Video contains audio: True
Source audio: /Users/chaitanyabhardwaj/vocult/files/test_audio.mp3 True
Audio file is valid: True
Output directory: /Users/chaitanyabhardwaj/vocult/files/emotion_output


In [11]:
from pathlib import Path
import json
import subprocess
import shutil

PROJECT = Path.home() / "vocult"
FILES = PROJECT / "files"

INPUT_VIDEO = FILES / "test.mp4"
SOURCE_AUDIO = FILES / "test_audio.mp3"
TRANSLATED_JSON = FILES / "test_audio_text_translated.json"
XTTS_DIR = FILES / "segments"
OUTPUT_DIR = FILES / "emotion_output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_WAV = OUTPUT_DIR / "original_speech.wav"
EMOTION_JSON = OUTPUT_DIR / "emotion_report.json"
DUBBED_WAV = OUTPUT_DIR / "dubbed_emotional.wav"
FINAL_VIDEO = OUTPUT_DIR / "vocult_emotional_dub.mp4"

assert INPUT_VIDEO.exists(), f"Missing video: {INPUT_VIDEO}"
assert SOURCE_AUDIO.exists(), f"Missing source audio: {SOURCE_AUDIO}"
assert TRANSLATED_JSON.exists(), f"Missing JSON: {TRANSLATED_JSON}"
assert XTTS_DIR.exists(), f"Missing XTTS directory: {XTTS_DIR}"

print("Configuration successful.")

Configuration successful.


In [12]:
subprocess.run([
    "ffmpeg", "-y",
    "-i", str(SOURCE_AUDIO),
    "-vn", "-ac", "1", "-ar", "16000",
    str(ORIGINAL_WAV)
], check=True)

print("Original audio saved:", ORIGINAL_WAV)

Original audio saved: /Users/chaitanyabhardwaj/vocult/files/emotion_output/original_speech.wav


ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
Input #0, mp3, from '/Users/chaitanyabhardwaj/vocult/files/test_audio.mp3':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso6iso2avc1mp41


In [3]:
from pathlib import Path

SOURCE_AUDIO = (
    Path.home() / "vocult" / "files" / "test_audio.mp3"
)

result = emotion_classifier.classify_file(str(SOURCE_AUDIO))

print("Predicted emotion:", result[3])
print("Prediction scores:", result[1])

CategoricalEncoder.expect_len was never called: assuming category count of 4 to be correct! Sanity check your encoder using `.expect_len`. Ensure that downstream code also uses the correct size. If you are sure this does not apply to you, use `.ignore_len`.


Predicted emotion: ['neu']
Prediction scores: tensor([1.0000])


In [6]:
from pathlib import Path
import subprocess

SOURCE_AUDIO = Path.home() / "vocult" / "files" / "test_audio.mp3"
OUTPUT_DIR = Path.home() / "vocult" / "files" / "emotion_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WAV_AUDIO = OUTPUT_DIR / "source_16k_mono.wav"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(SOURCE_AUDIO),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(WAV_AUDIO),
    ],
    check=True,
    capture_output=True,
    text=True,
)

print("Converted audio:", WAV_AUDIO)

Converted audio: /Users/chaitanyabhardwaj/vocult/files/emotion_output/source_16k_mono.wav


In [7]:
import wave
import json
import torch
from pathlib import Path

WAV_AUDIO = OUTPUT_DIR / "source_16k_mono.wav"
window_seconds = 3
sample_rate = 16000

with wave.open(str(WAV_AUDIO), "rb") as wav:
    assert wav.getnchannels() == 1, "Audio must be mono"
    assert wav.getframerate() == sample_rate, "Audio must be 16 kHz"

    total_frames = wav.getnframes()
    audio_bytes = wav.readframes(total_frames)

# Convert 16-bit PCM bytes into a normalized PyTorch tensor
waveform = torch.frombuffer(
    bytearray(audio_bytes), dtype=torch.int16
).float() / 32768.0

window_samples = window_seconds * sample_rate
emotion_results = []
temp_path = OUTPUT_DIR / "temp_emotion_chunk.wav"

for start_sample in range(0, len(waveform), window_samples):
    end_sample = min(start_sample + window_samples, len(waveform))

    if end_sample - start_sample < sample_rate:
        continue

    chunk = waveform[start_sample:end_sample]

    with wave.open(str(temp_path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)

        pcm = (
            chunk.clamp(-1, 1) * 32767
        ).short().numpy().tobytes()

        wav.writeframes(pcm)

    result = emotion_classifier.classify_file(str(temp_path))

    emotion_results.append({
        "start": round(start_sample / sample_rate, 2),
        "end": round(end_sample / sample_rate, 2),
        "emotion": str(result[3][0]) if isinstance(
            result[3], (list, tuple)
        ) else str(result[3]),
        "score": float(result[1].max().item())
        if torch.is_tensor(result[1]) else None,
    })

output_path = OUTPUT_DIR / "emotion_timeline.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(emotion_results, f, indent=2, ensure_ascii=False)

temp_path.unlink(missing_ok=True)

print("Saved:", output_path)
print("Windows detected:", len(emotion_results))

for item in emotion_results:
    print(item)

Saved: /Users/chaitanyabhardwaj/vocult/files/emotion_output/emotion_timeline.json
Windows detected: 46
{'start': 0.0, 'end': 3.0, 'emotion': 'neu', 'score': 1.0}
{'start': 3.0, 'end': 6.0, 'emotion': 'neu', 'score': 1.0}
{'start': 6.0, 'end': 9.0, 'emotion': 'neu', 'score': 1.0}
{'start': 9.0, 'end': 12.0, 'emotion': 'neu', 'score': 1.0}
{'start': 12.0, 'end': 15.0, 'emotion': 'ang', 'score': 0.9999462366104126}
{'start': 15.0, 'end': 18.0, 'emotion': 'neu', 'score': 1.0}
{'start': 18.0, 'end': 21.0, 'emotion': 'neu', 'score': 0.9999966621398926}
{'start': 21.0, 'end': 24.0, 'emotion': 'neu', 'score': 1.0}
{'start': 24.0, 'end': 27.0, 'emotion': 'neu', 'score': 1.0}
{'start': 27.0, 'end': 30.0, 'emotion': 'neu', 'score': 1.0}
{'start': 30.0, 'end': 33.0, 'emotion': 'neu', 'score': 0.9999998807907104}
{'start': 33.0, 'end': 36.0, 'emotion': 'neu', 'score': 0.9999961853027344}
{'start': 36.0, 'end': 39.0, 'emotion': 'ang', 'score': 0.9999113082885742}
{'start': 39.0, 'end': 42.0, 'emotio

In [8]:
import json
from pathlib import Path
from collections import Counter

BASE_DIR = Path.home() / "vocult" / "files"
OUTPUT_DIR = BASE_DIR / "emotion_output"

emotion_file = OUTPUT_DIR / "emotion_timeline.json"

with open(emotion_file, "r", encoding="utf-8") as f:
    emotion_timeline = json.load(f)

if not emotion_timeline:
    raise ValueError(
        "No emotion windows were generated. Check the source audio and classifier."
    )

print("Total windows:", len(emotion_timeline))
print("\nEmotion timeline:")

for item in emotion_timeline:
    print(
        f"{item['start']:>6.2f}s - {item['end']:>6.2f}s | "
        f"{item['emotion']} | score={item.get('score')}"
    )

counts = Counter(item["emotion"] for item in emotion_timeline)
print("\nEmotion counts:", dict(counts))

Total windows: 46

Emotion timeline:
  0.00s -   3.00s | neu | score=1.0
  3.00s -   6.00s | neu | score=1.0
  6.00s -   9.00s | neu | score=1.0
  9.00s -  12.00s | neu | score=1.0
 12.00s -  15.00s | ang | score=0.9999462366104126
 15.00s -  18.00s | neu | score=1.0
 18.00s -  21.00s | neu | score=0.9999966621398926
 21.00s -  24.00s | neu | score=1.0
 24.00s -  27.00s | neu | score=1.0
 27.00s -  30.00s | neu | score=1.0
 30.00s -  33.00s | neu | score=0.9999998807907104
 33.00s -  36.00s | neu | score=0.9999961853027344
 36.00s -  39.00s | ang | score=0.9999113082885742
 39.00s -  42.00s | neu | score=0.9999157190322876
 42.00s -  45.00s | ang | score=0.9956823587417603
 45.00s -  48.00s | neu | score=0.9742611646652222
 48.00s -  51.00s | neu | score=0.999853253364563
 51.00s -  54.00s | neu | score=0.9982942938804626
 54.00s -  57.00s | neu | score=1.0
 57.00s -  60.00s | neu | score=1.0
 60.00s -  63.00s | hap | score=0.9999721050262451
 63.00s -  66.00s | neu | score=1.0
 66.00s

In [9]:
from collections import Counter

smoothed_timeline = []

for i, item in enumerate(emotion_timeline):
    left = max(0, i - 1)
    right = min(len(emotion_timeline), i + 2)

    nearby_labels = [
        x["emotion"] for x in emotion_timeline[left:right]
    ]

    majority_emotion = Counter(nearby_labels).most_common(1)[0][0]

    smoothed_timeline.append({
        **item,
        "emotion": majority_emotion
    })

smoothed_file = OUTPUT_DIR / "emotion_timeline_smoothed.json"

with open(smoothed_file, "w", encoding="utf-8") as f:
    json.dump(smoothed_timeline, f, indent=2, ensure_ascii=False)

print("Smoothed emotion timeline saved to:", smoothed_file)

Smoothed emotion timeline saved to: /Users/chaitanyabhardwaj/vocult/files/emotion_output/emotion_timeline_smoothed.json


In [10]:
TRANSLATED_JSON = BASE_DIR / "test_audio_text_translated.json"

with open(TRANSLATED_JSON, "r", encoding="utf-8") as f:
    translated_data = json.load(f)

print("JSON type:", type(translated_data).__name__)

if isinstance(translated_data, dict):
    print("Top-level keys:", list(translated_data.keys()))
    print(json.dumps(translated_data, indent=2, ensure_ascii=False)[:3000])
elif isinstance(translated_data, list):
    print("Number of segments:", len(translated_data))
    print(json.dumps(translated_data[:3], indent=2, ensure_ascii=False))

JSON type: dict
Top-level keys: ['source_language', 'target_language', 'segments', 'metadata']
{
  "source_language": "eng_Latn",
  "target_language": "hin_Deva",
  "segments": [
    {
      "start": 0.291,
      "end": 1.893,
      "speaker": "SPEAKER_00",
      "original_text": "As humans, we understand each other.",
      "translated_text": "मनुष्य के रूप में, हम एक दूसरे को समझते हैं।"
    },
    {
      "start": 2.273,
      "end": 2.734,
      "speaker": "SPEAKER_00",
      "original_text": "Roger that.",
      "translated_text": "समझ में आ गया."
    },
    {
      "start": 3.174,
      "end": 3.815,
      "speaker": "SPEAKER_01",
      "original_text": "I feel ya.",
      "translated_text": "मैं तुम्हें महसूस करता हूँ."
    },
    {
      "start": 4.215,
      "end": 5.016,
      "speaker": "SPEAKER_01",
      "original_text": "Aye-aye, Captain.",
      "translated_text": "हाँ-हाँ, कप्तान."
    },
    {
      "start": 5.697,
      "end": 6.918,
      "speaker": "SPEAKER_01",
   

In [11]:
from pathlib import Path
import json

BASE_DIR = Path.home() / "vocult" / "files"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(BASE_DIR / "test_audio_text_translated.json", "r", encoding="utf-8") as f:
    translated_data = json.load(f)

print(json.dumps(translated_data[:2] if isinstance(translated_data, list) else translated_data,
                 indent=2, ensure_ascii=False)[:3000])

{
  "source_language": "eng_Latn",
  "target_language": "hin_Deva",
  "segments": [
    {
      "start": 0.291,
      "end": 1.893,
      "speaker": "SPEAKER_00",
      "original_text": "As humans, we understand each other.",
      "translated_text": "मनुष्य के रूप में, हम एक दूसरे को समझते हैं।"
    },
    {
      "start": 2.273,
      "end": 2.734,
      "speaker": "SPEAKER_00",
      "original_text": "Roger that.",
      "translated_text": "समझ में आ गया."
    },
    {
      "start": 3.174,
      "end": 3.815,
      "speaker": "SPEAKER_01",
      "original_text": "I feel ya.",
      "translated_text": "मैं तुम्हें महसूस करता हूँ."
    },
    {
      "start": 4.215,
      "end": 5.016,
      "speaker": "SPEAKER_01",
      "original_text": "Aye-aye, Captain.",
      "translated_text": "हाँ-हाँ, कप्तान."
    },
    {
      "start": 5.697,
      "end": 6.918,
      "speaker": "SPEAKER_01",
      "original_text": "Wait, what are we talking about?",
      "translated_text": "प्रतीक्षा, हम

In [1]:
import torch
from transformers.pytorch_utils import isin_mps_friendly
from TTS.api import TTS

print("PyTorch:", torch.__version__)
print("XTTS import successful!")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0
XTTS import successful!


In [2]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2"
).to(DEVICE)

print("XTTS loaded on:", DEVICE)

XTTS loaded on: mps


In [4]:
import json
from pathlib import Path

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

TRANSLATED_JSON = BASE_DIR / "test_audio_text_translated.json"

with open(TRANSLATED_JSON, "r", encoding="utf-8") as f:
    translated_data = json.load(f)

print("Translated JSON loaded successfully!")
print("Data type:", type(translated_data))

if isinstance(translated_data, dict):
    print("Available keys:", translated_data.keys())
elif isinstance(translated_data, list):
    print("Number of segments:", len(translated_data))

print("First item:", translated_data[0] if isinstance(translated_data, list)
      else list(translated_data.items())[:1])

Translated JSON loaded successfully!
Data type: <class 'dict'>
Available keys: dict_keys(['source_language', 'target_language', 'segments', 'metadata'])
First item: [('source_language', 'eng_Latn')]


In [5]:
if isinstance(translated_data, list):
    segments = translated_data
elif isinstance(translated_data, dict):
    segments = translated_data.get("segments", [])

if not segments:
    raise ValueError("No segments found. Check the JSON structure.")

SEGMENTS_DIR = BASE_DIR / "tts_segments"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Found {len(segments)} translated segments.")
print("First translated sentence:", segments[0].get("translated_text"))

Found 42 translated segments.
First translated sentence: मनुष्य के रूप में, हम एक दूसरे को समझते हैं।


In [8]:
from pathlib import Path
import subprocess
import shutil

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")
reference_audio = BASE_DIR / "speaker_reference.wav"
converted_audio = BASE_DIR / "speaker_reference_24k.wav"

if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpeg is not installed or is not on PATH.")

subprocess.run([
    "ffmpeg", "-y",
    "-i", str(reference_audio),
    "-ac", "1",
    "-ar", "24000",
    str(converted_audio)
], check=True)

print("Converted reference audio:", converted_audio)

Converted reference audio: /Users/chaitanyabhardwaj/vocult/files/speaker_reference_24k.wav


ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
[aist#0:0/pcm_s16le @ 0x7aa7008180] Guessed Channel Layout: mono
Input #0, wav, from '/Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav':
  Metadata:
    artist          : Dee

In [10]:
audio, sample_rate = sf.read(converted_audio, dtype="float32")

print("Audio shape:", audio.shape)
print("Sample rate:", sample_rate)
print("Audio loaded successfully!")%pip install edge-tts soundfile numpy imageio-ffmpeg

Audio shape: (240000,)
Sample rate: 24000
Audio loaded successfully!


In [11]:
%pip install edge-tts soundfile numpy imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [edge-tts]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import asyncio
import json
import re
import subprocess
import shutil
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

VIDEO_PATH = BASE_DIR / "test_video.mp4"
JSON_PATH = BASE_DIR / "test_audio_text_translated.json"

OUTPUT_DIR = BASE_DIR / "output"
SEGMENTS_DIR = BASE_DIR / "hindi_tts_segments"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_AUDIO = OUTPUT_DIR / "translated_speech_hindi.wav"
FINAL_VIDEO = OUTPUT_DIR / "test_video_hindi.mp4"

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
SAMPLE_RATE = 24000
VOICE = "hi-IN-SwaraNeural"

# --------------------------------------------------
# 2. Validate inputs and load translated JSON
# --------------------------------------------------
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

if not JSON_PATH.exists():
    raise FileNotFoundError(f"Translated JSON not found: {JSON_PATH}")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    translated_data = json.load(f)

if isinstance(translated_data, list):
    segments = translated_data
elif isinstance(translated_data, dict):
    segments = translated_data.get("segments", [])
else:
    segments = []

if not segments:
    raise ValueError("No translated segments found in the JSON.")

print("Video:", VIDEO_PATH)
print("Translated segments:", len(segments))
print("Hindi voice:", VOICE)

# --------------------------------------------------
# 3. Get the original video duration
# --------------------------------------------------
probe = subprocess.run(
    [FFMPEG, "-i", str(VIDEO_PATH)],
    capture_output=True,
    text=True
)

match = re.search(
    r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
    probe.stderr
)

if not match:
    raise RuntimeError(
        "Could not determine video duration. Check that the MP4 is valid."
    )

video_duration = (
    int(match.group(1)) * 3600
    + int(match.group(2)) * 60
    + float(match.group(3))
)

print(f"Video duration: {video_duration:.2f} seconds")

# --------------------------------------------------
# 4. Generate Hindi speech for every segment
# --------------------------------------------------
async def generate_hindi_segments():
    generated = []

    for i, segment in enumerate(segments):
        text = (
            segment.get("translated_text")
            or segment.get("text")
            or ""
        ).strip()

        if not text:
            continue

        start = float(segment.get("start", 0))
        if start < 0:
            start = 0

        mp3_path = SEGMENTS_DIR / f"segment_{i:03d}.mp3"
        wav_path = SEGMENTS_DIR / f"segment_{i:03d}.wav"

        # Generate Hindi speech
        communicate = edge_tts.Communicate(
            text=text,
            voice=VOICE
        )
        await communicate.save(str(mp3_path))

        # Convert speech to 24 kHz mono WAV
        subprocess.run(
            [
                FFMPEG, "-y",
                "-i", str(mp3_path),
                "-ac", "1",
                "-ar", str(SAMPLE_RATE),
                "-c:a", "pcm_s16le",
                str(wav_path)
            ],
            check=True,
            capture_output=True
        )

        audio, sr = sf.read(wav_path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        generated.append({
            "index": i,
            "text": text,
            "start": start,
            "audio": audio
        })

        print(f"Generated {len(generated)}: {text[:65]}")

    return generated


generated_segments = await generate_hindi_segments()

if not generated_segments:
    raise RuntimeError("No speech segments were generated.")

print(f"\nSuccessfully generated {len(generated_segments)} Hindi segments.")

# --------------------------------------------------
# 5. Align generated speech with original timestamps
# --------------------------------------------------
total_samples = int(video_duration * SAMPLE_RATE)
timeline = np.zeros(total_samples, dtype=np.float32)

for item in generated_segments:
    start_sample = int(item["start"] * SAMPLE_RATE)
    audio = item["audio"]

    if start_sample >= total_samples:
        continue

    # Keep audio within the video's duration
    end_sample = min(
        start_sample + len(audio),
        total_samples
    )
    audio = audio[:end_sample - start_sample]

    # Mix segments at their original start times
    timeline[start_sample:end_sample] += audio

# Prevent clipping after mixing
peak = np.max(np.abs(timeline))
if peak > 0.99:
    timeline = timeline * (0.99 / peak)

sf.write(FINAL_AUDIO, timeline, SAMPLE_RATE)

print("Hindi audio saved:", FINAL_AUDIO)

# --------------------------------------------------
# 6. Merge Hindi audio with the original video
# --------------------------------------------------
subprocess.run(
    [
        FFMPEG, "-y",
        "-i", str(VIDEO_PATH),
        "-i", str(FINAL_AUDIO),
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-t", str(video_duration),
        "-movflags", "+faststart",
        str(FINAL_VIDEO)
    ],
    check=True
)

print("\nHindi dubbing complete!")
print("Final video:", FINAL_VIDEO)
print("Video exists:", FINAL_VIDEO.exists())

Video: /Users/chaitanyabhardwaj/vocult/files/test_video.mp4
Translated segments: 42
Hindi voice: hi-IN-SwaraNeural
Video duration: 138.00 seconds
Generated 1: मनुष्य के रूप में, हम एक दूसरे को समझते हैं।
Generated 2: समझ में आ गया.
Generated 3: मैं तुम्हें महसूस करता हूँ.
Generated 4: हाँ-हाँ, कप्तान.
Generated 5: प्रतीक्षा, हम क्या बात कर रहे हैं?
Generated 6: खैर, ज्यादातर समय.
Generated 7: यहां तक कि अगर हम कुछ शब्दों को स्पष्ट रूप से नहीं सुनते हैं, तो 
Generated 8: अब, मशीनें भी हमें समझ सकती हैं।
Generated 9: हमारी घड़ियाँ, कारें, फोन, ड्राइव-थ्रू सभी हमारे शब्दों को संसाधि
Generated 10: कुछ पागल जादू की तरह लगता है, है ना?
Generated 11: खैर, यह वास्तव में जादू नहीं है, लेकिन यह काफी भयानक है.
Generated 12: और मूल अद्भुत सॉस जो यह सब संभव बनाता है भाषण पहचान है।
Generated 13: यह क्या है?
Generated 14: संक्षिप्त उत्तर?
Generated 15: यह सॉफ्टवेयर है जो हमें ऑडियो को उपयोग करने योग्य, संरचित डेटा मे
Generated 16: दूसरे शब्दों में, ऑडियो अंदर आता है, पाठ बाहर आता है।
Generated 17: फि

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 13.1.6 (clang-1316.0.21.2.5)
  configuration: --prefix=/Volumes/tempdisk/sw --extra-cflags=-fno-stack-check --arch=arm64 --cc=/usr/bin/clang --enable-gpl --enable-libvmaf --enable-libopenjpeg --enable-libopus --enable-libmp3lame --enable-libx264 --enable-libx265 --enable-libvpx --enable-libwebp --enable-libass --enable-libfreetype --enable-fontconfig --enable-libtheora --enable-libvorbis --enable-libsnappy --enable-libaom --enable-libvidstab --enable-libzimg --enable-libsvtav1 --enable-libharfbuzz --enable-libkvazaar --pkg-config-flags=--static --enable-ffplay --enable-postproc --enable-neon --enable-runtime-cpudetect --disable-indev=qtkit --disable-indev=x11grab_xcb
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8. 


Hindi dubbing complete!
Final video: /Users/chaitanyabhardwaj/vocult/files/output/test_video_hindi.mp4
Video exists: True


[mp4 @ 0x123608c90] Starting second pass: moving the moov atom to the beginning of the file123x    
[out#0/mp4 @ 0x600000e8c3c0] video:31155KiB audio:1797KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.293993%
frame= 4140 fps=3589 q=-1.0 Lsize=   33049KiB time=00:02:17.96 bitrate=1962.3kbits/s speed= 120x    
[aac @ 0x12360a3b0] Qavg: 65499.156
